# 01. 가설 검증 및 DPO 데이터 생성

동일한 금융 사기 상황을 **일반 페르소나(20~30대, 디지털 활용 능력 높음)**와 **소수자 페르소나(65세 이상, 디지털 활용 능력 낮음)**로 질문했을 때
LLM 응답의 Fluff 지수에 차이가 있는지 측정하고, 그 결과로 DPO 학습용 선호 데이터(prompt / chosen / rejected)를 만듭니다.

| 섹션 | 내용 |
|---|---|
| 1 | 시나리오·페르소나 응답 생성 + ConCISE 방식 Fluff 지수 계산 |
| 2 | 여러 번 나눠 실행한 결과 JSON 병합 |
| 3 | DPO 데이터 생성 (v1: 후보 다중 생성 / v2: 단일 호출, 실제 데이터셋에 사용) |

> **API 키 설정**: 이 노트북은 키를 코드에 적지 않고 Colab Secrets에서 불러옵니다.
> Colab 왼쪽 🔑(Secrets) 탭에 `GEMINI_API_KEY`라는 이름으로 키를 등록하고 노트북 접근을 허용하세요.

In [ ]:
!pip -q install "google-genai>=1.64.0,<2.0.0" "google-auth==2.47.0"

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

## 1. 가설 검증: 페르소나별 응답 생성 + Fluff 지수

- 10개 금융 사기 유형을 순환하며 시나리오를 생성하고, 두 페르소나에 대해 각각 원본 응답을 생성합니다.
- 각 응답을 추상적 요약(P1) · 추출적 요약(P2) · 불필요 단어 제거(P3)로 압축한 뒤
  `Fluff Index = 1 − mean(P1, P2, P3 토큰 수 / 원문 토큰 수)` 로 계산합니다.
- 결과는 `experiment_standard_results.json`, `experiment_minority_results.json`에 저장됩니다.

In [ ]:
from google.colab import userdata
import os
import json
import time
import re
import random
from typing import Dict, Any, List, Optional

from google import genai
from google.genai import types

os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["http_proxy"] = ""
os.environ["https_proxy"] = ""
os.environ["NO_PROXY"] = "*"


GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_GEN = "gemini-2.5-flash-lite"
MODEL_COMP = "gemini-2.5-flash-lite"

FILE_STANDARD = "experiment_standard_results.json"
FILE_MINORITY = "experiment_minority_results.json"
FILE_ERROR_LOG = "experiment_error_log.jsonl"


API_TIER = "paid"

if API_TIER == "free":
    MAX_RETRIES = 6
    RETRY_SLEEP_SECONDS = 8
    REQUEST_GAP_SECONDS = 5.0
    PAIR_GAP_SECONDS = 2.0
else:
    MAX_RETRIES = 6
    RETRY_SLEEP_SECONDS = 3
    REQUEST_GAP_SECONDS = 0.8
    PAIR_GAP_SECONDS = 0.8


SAFETY_SETTINGS = [
    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
]


SCENARIO_RESPONSE_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "scenario_id": types.Schema(type=types.Type.STRING),
        "fraud_type": types.Schema(type=types.Type.STRING),
        "fraud_situation": types.Schema(type=types.Type.STRING),
        "standard_persona": types.Schema(
            type=types.Type.OBJECT,
            properties={
                "context": types.Schema(type=types.Type.STRING),
                "content": types.Schema(type=types.Type.STRING),
            },
            required=["context", "content"],
        ),
        "minority_persona": types.Schema(
            type=types.Type.OBJECT,
            properties={
                "context": types.Schema(type=types.Type.STRING),
                "content": types.Schema(type=types.Type.STRING),
            },
            required=["context", "content"],
        ),
        "question": types.Schema(type=types.Type.STRING),
    },
    required=[
        "scenario_id",
        "fraud_type",
        "fraud_situation",
        "standard_persona",
        "minority_persona",
        "question",
    ],
)

COMPRESSION_RESPONSE_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "p1_abstractive_text": types.Schema(type=types.Type.STRING),
        "p2_extractive_text": types.Schema(type=types.Type.STRING),
        "p3_word_removed_text": types.Schema(type=types.Type.STRING),
    },
    required=[
        "p1_abstractive_text",
        "p2_extractive_text",
        "p3_word_removed_text",
    ],
)


SCENARIO_GENERATION_CONFIG = {
    "response_mime_type": "application/json",
    "response_schema": SCENARIO_RESPONSE_SCHEMA,
    "temperature": 0.4,
    "top_p": 0.9,
    "max_output_tokens": 4096,
}

ORIGINAL_RESPONSE_CONFIG = {
    "temperature": 0.2,
    "top_p": 1.0,
    "max_output_tokens": 4096,
}

MERGED_COMPRESSION_CONFIG = {
    "response_mime_type": "application/json",
    "response_schema": COMPRESSION_RESPONSE_SCHEMA,
    "temperature": 0.0,
    "top_p": 1.0,
    "max_output_tokens": 12288,
}


def make_generation_config(
    config_dict: Dict[str, Any],
    system_instruction: Optional[str] = None,
):
    kwargs = {
        "temperature": config_dict.get("temperature"),
        "top_p": config_dict.get("top_p"),
        "max_output_tokens": config_dict.get("max_output_tokens"),
        "safety_settings": SAFETY_SETTINGS,
    }

    if config_dict.get("response_mime_type"):
        kwargs["response_mime_type"] = config_dict["response_mime_type"]

    if config_dict.get("response_schema"):
        kwargs["response_schema"] = config_dict["response_schema"]

    if system_instruction:
        kwargs["system_instruction"] = system_instruction

    return types.GenerateContentConfig(**kwargs)


_LAST_API_CALL_TIME = 0.0


def rate_limited_sleep() -> None:
    global _LAST_API_CALL_TIME

    now = time.time()
    elapsed = now - _LAST_API_CALL_TIME
    wait = REQUEST_GAP_SECONDS - elapsed

    if wait > 0:
        time.sleep(wait)

    _LAST_API_CALL_TIME = time.time()


def get_retry_sleep_seconds(attempt: int) -> float:
    base = RETRY_SLEEP_SECONDS * (2 ** (attempt - 1))
    jitter = random.uniform(0, 2.0)
    return base + jitter


class AuthError(Exception):
    pass


def extract_response_text(response, stage_name: str) -> str:
    try:
        text = response.text
    except Exception as e:
        raise RuntimeError(f"{stage_name}: response.text extraction failed: {e}") from e

    if not isinstance(text, str) or not text.strip():
        raise ValueError(f"{stage_name}: empty model response.")

    return text.strip()


def clean_json_text(text: str) -> str:
    cleaned = text.strip()

    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()

    return cleaned


def load_json_array(path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(path):
        return []

    if os.path.getsize(path) == 0:
        return []

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON array.")

    return data


def append_json_array_atomic(path: str, node: Dict[str, Any]) -> None:
    data = load_json_array(path)
    data.append(node)

    temp_path = f"{path}.tmp"

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    os.replace(temp_path, path)


def initialize_result_files(overwrite: bool = False) -> None:
    for path in [FILE_STANDARD, FILE_MINORITY]:
        if overwrite or not os.path.exists(path):
            with open(path, "w", encoding="utf-8") as f:
                json.dump([], f, ensure_ascii=False, indent=2)

    if overwrite and os.path.exists(FILE_ERROR_LOG):
        os.remove(FILE_ERROR_LOG)


def log_error(
    stage: str,
    index: int,
    error: Exception,
    extra: Dict[str, Any] = None,
) -> None:
    error_node = {
        "stage": stage,
        "index": index,
        "error_type": type(error).__name__,
        "error_message": str(error),
        "extra": extra or {},
    }

    with open(FILE_ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(error_node, ensure_ascii=False) + "\n")


def generate_text_with_retry(
    model_name: str,
    prompt: str,
    generation_config: Dict[str, Any],
    stage_name: str,
    system_instruction: Optional[str] = None,
    max_retries: int = MAX_RETRIES,
) -> str:
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            rate_limited_sleep()

            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=make_generation_config(
                    generation_config,
                    system_instruction=system_instruction,
                ),
            )

            return extract_response_text(response, stage_name)

        except Exception as e:
            err_str = str(e)

            if "401" in err_str or "UNAUTHENTICATED" in err_str:
                raise AuthError(
                    f"{stage_name}: API 키 인증 실패입니다. "
                    f"GEMINI_API_KEY가 유효한지 확인하세요. 원본 오류: {e}"
                ) from e

            last_error = e
            print(f"      ⚠️ {stage_name} 실패 ({attempt}/{max_retries}): {e}")

            if attempt < max_retries:
                sleep_time = get_retry_sleep_seconds(attempt)
                print(f"      ↳ {sleep_time:.1f}초 후 재시도합니다.")
                time.sleep(sleep_time)

    raise RuntimeError(f"{stage_name} failed after {max_retries} retries: {last_error}")


def get_gemini_token_count(text: str) -> int:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Token count failed: input text is empty.")

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            rate_limited_sleep()

            result = client.models.count_tokens(
                model=MODEL_COMP,
                contents=text,
            )

            token_count = getattr(result, "total_tokens", None)

            if token_count is None:
                raise RuntimeError("Gemini token counter did not return total_tokens.")

            return int(token_count)

        except Exception as e:
            err_str = str(e)

            if "401" in err_str or "UNAUTHENTICATED" in err_str:
                raise AuthError(
                    f"Token count: API 키 인증 실패입니다. 원본 오류: {e}"
                ) from e

            last_error = e
            print(f"      ⚠️ Token count 실패 ({attempt}/{MAX_RETRIES}): {e}")

            if attempt < MAX_RETRIES:
                sleep_time = get_retry_sleep_seconds(attempt)
                print(f"      ↳ {sleep_time:.1f}초 후 재시도합니다.")
                time.sleep(sleep_time)

    raise RuntimeError(f"Gemini token count failed after retries: {last_error}")


def validate_pair_data(pair_data: Dict[str, Any], index: int) -> Dict[str, Any]:
    required_top_keys = [
        "scenario_id",
        "fraud_type",
        "fraud_situation",
        "standard_persona",
        "minority_persona",
        "question",
    ]

    for key in required_top_keys:
        if key not in pair_data:
            raise ValueError(f"Scenario JSON missing required key: {key}")

    for key in ["scenario_id", "fraud_type", "fraud_situation", "question"]:
        if not isinstance(pair_data[key], str) or not pair_data[key].strip():
            raise ValueError(f"Scenario JSON field must be non-empty string: {key}")

    for persona_key in ["standard_persona", "minority_persona"]:
        persona = pair_data[persona_key]

        if not isinstance(persona, dict):
            raise ValueError(f"{persona_key} must be an object.")

        for sub_key in ["context", "content"]:
            if sub_key not in persona:
                raise ValueError(f"{persona_key} missing required key: {sub_key}")

            if not isinstance(persona[sub_key], str) or not persona[sub_key].strip():
                raise ValueError(f"{persona_key}.{sub_key} must be non-empty string.")

    return pair_data


SCENARIO_JSON_MAX_ATTEMPTS = 2


def generate_pair_data(index: int) -> Dict[str, Any]:
    prompt = get_single_pair_prompt(index)
    last_error = None
    last_raw_text = None

    for attempt in range(1, SCENARIO_JSON_MAX_ATTEMPTS + 1):
        try:
            raw_text = generate_text_with_retry(
                model_name=MODEL_GEN,
                prompt=prompt,
                generation_config=SCENARIO_GENERATION_CONFIG,
                stage_name=f"Scenario generation #{index} JSON attempt {attempt}",
            )

            last_raw_text = raw_text
            pair_data = json.loads(clean_json_text(raw_text))
            return validate_pair_data(pair_data, index)

        except json.JSONDecodeError as e:
            last_error = e
            print(
                f"      ⚠️ Scenario JSON parsing 실패 "
                f"({attempt}/{SCENARIO_JSON_MAX_ATTEMPTS}): {e}"
            )

            if attempt < SCENARIO_JSON_MAX_ATTEMPTS:
                sleep_time = get_retry_sleep_seconds(attempt)
                print(f"      ↳ JSON 재생성을 위해 {sleep_time:.1f}초 후 재시도합니다.")
                time.sleep(sleep_time)

        except AuthError:
            raise

        except Exception as e:
            last_error = e
            print(
                f"      ⚠️ Scenario generation/validation 실패 "
                f"({attempt}/{SCENARIO_JSON_MAX_ATTEMPTS}): {e}"
            )

            if attempt < SCENARIO_JSON_MAX_ATTEMPTS:
                sleep_time = get_retry_sleep_seconds(attempt)
                print(f"      ↳ {sleep_time:.1f}초 후 재시도합니다.")
                time.sleep(sleep_time)

    raise ValueError(
        f"Scenario JSON failed after {SCENARIO_JSON_MAX_ATTEMPTS} attempts: {last_error}\n"
        f"Last raw text:\n{last_raw_text}"
    )


COMPRESSION_JSON_MAX_ATTEMPTS = 2


def get_merged_compression_prompt(orig_text: str) -> str:
    return f"""You will be given ONE original response. Produce THREE different compressed
versions of it by applying three independent compression strategies to the
SAME original response. Output ONLY a JSON object with the three keys
described below.

ORIGINAL RESPONSE:
\"\"\"
{orig_text}
\"\"\"

STRATEGY 1 -> key "p1_abstractive_text" (Abstractive summary):
Summarize the response in as few words as possible while preserving ALL
informational content. Do not omit any facts, numbers, steps, warnings, or
claims. Output only the compressed answer.

STRATEGY 2 -> key "p2_extractive_text" (Extractive summary):
From the original response, copy only the sentences that contain essential
information. Do NOT paraphrase or rewrite. Copy the sentences verbatim,
exactly as they appear. Omit filler sentences, compliments, transitions, and
redundant restatements. If all sentences are essential, copy all of them.
Never output an empty string.

STRATEGY 3 -> key "p3_word_removed_text" (Word-removed compression):
Remove as many words as possible from the original response while preserving
its complete meaning.
- Keep all facts, numbers, claims, steps, warnings, and logical structure.
- Remove filler words, redundant adjectives, unnecessary adverbs, hedging
  phrases, politeness expressions, transitional padding, and robotic flattery.
- Do NOT rewrite or restructure sentences.
- Never output an empty string.

Return a JSON object with exactly these three keys:
"p1_abstractive_text", "p2_extractive_text", "p3_word_removed_text".
Each value must be the corresponding compressed text as a plain string.
"""


def validate_compression_data(data: Dict[str, Any]) -> Dict[str, Any]:
    required_keys = [
        "p1_abstractive_text",
        "p2_extractive_text",
        "p3_word_removed_text",
    ]

    for key in required_keys:
        if key not in data:
            raise ValueError(f"Compression JSON missing required key: {key}")

        if not isinstance(data[key], str) or not data[key].strip():
            raise ValueError(f"Compression JSON field must be non-empty string: {key}")

    return data


def get_merged_compression(orig_text: str, stage_name: str) -> Dict[str, Any]:
    prompt = get_merged_compression_prompt(orig_text)
    last_error = None
    last_raw_text = None

    for attempt in range(1, COMPRESSION_JSON_MAX_ATTEMPTS + 1):
        try:
            raw_text = generate_text_with_retry(
                model_name=MODEL_COMP,
                prompt=prompt,
                generation_config=MERGED_COMPRESSION_CONFIG,
                stage_name=f"{stage_name} JSON attempt {attempt}",
                system_instruction=(
                    "You are a precise text-compression engine for academic measurement. "
                    "Apply each compression strategy exactly as specified and return valid JSON only."
                ),
            )

            last_raw_text = raw_text
            data = json.loads(clean_json_text(raw_text))
            return validate_compression_data(data)

        except json.JSONDecodeError as e:
            last_error = e
            print(
                f"      ⚠️ Compression JSON parsing 실패 "
                f"({attempt}/{COMPRESSION_JSON_MAX_ATTEMPTS}): {e}"
            )

            if attempt < COMPRESSION_JSON_MAX_ATTEMPTS:
                sleep_time = get_retry_sleep_seconds(attempt)
                print(f"      ↳ JSON 재생성을 위해 {sleep_time:.1f}초 후 재시도합니다.")
                time.sleep(sleep_time)

        except AuthError:
            raise

        except Exception as e:
            last_error = e
            print(
                f"      ⚠️ Compression generation/validation 실패 "
                f"({attempt}/{COMPRESSION_JSON_MAX_ATTEMPTS}): {e}"
            )

            if attempt < COMPRESSION_JSON_MAX_ATTEMPTS:
                sleep_time = get_retry_sleep_seconds(attempt)
                print(f"      ↳ {sleep_time:.1f}초 후 재시도합니다.")
                time.sleep(sleep_time)

    raise ValueError(
        f"Compression JSON failed after {COMPRESSION_JSON_MAX_ATTEMPTS} attempts: {last_error}\n"
        f"Last raw text:\n{last_raw_text}"
    )


def calculate_metrics(original: str, p1: str, p2: str, p3: str) -> Dict[str, Any]:
    tok_orig = get_gemini_token_count(original)
    tok_p1 = get_gemini_token_count(p1)
    tok_p2 = get_gemini_token_count(p2)
    tok_p3 = get_gemini_token_count(p3)

    if tok_orig <= 0:
        raise ValueError("Original response token count must be greater than zero.")

    compressed_tokens = {
        "p1_abstractive": tok_p1,
        "p2_extractive": tok_p2,
        "p3_word_removed": tok_p3,
    }

    for name, count in compressed_tokens.items():
        if count <= 0:
            raise ValueError(f"{name} token count is zero or negative.")

        if count > tok_orig:
            raise ValueError(
                f"{name} is longer than original. "
                f"original={tok_orig}, {name}={count}"
            )

    score_1 = tok_p1 / tok_orig
    score_2 = tok_p2 / tok_orig
    score_3 = tok_p3 / tok_orig

    mean_score = (score_1 + score_2 + score_3) / 3
    fluff_index = 1 - mean_score

    if not (0 <= fluff_index <= 1):
        raise ValueError(f"Fluff index out of range: {fluff_index}")

    return {
        "token_counts": {
            "original": tok_orig,
            "p1_abstractive": tok_p1,
            "p2_extractive": tok_p2,
            "p3_word_removed": tok_p3,
        },
        "scores": {
            "score_1": round(score_1, 4),
            "score_2": round(score_2, 4),
            "score_3": round(score_3, 4),
        },
        "fluff_index": round(fluff_index, 4),
    }


def get_single_pair_prompt(index: int) -> str:
    fraud_types = [
        "Smishing",
        "Voice Phishing",
        "Fake Delivery Notification",
        "Tax Refund Scam",
        "Family Impersonation",
        "Unauthorized Credit Card Opening",
        "Government Subsidy Scam",
        "Financial Institution Impersonation",
        "Illegal Investment Solicitation",
        "Identity Theft",
    ]

    selected_type = fraud_types[(index - 1) % len(fraud_types)]

    return f"""
You are a financial fraud scenario designer for academic research.
Generate exactly ONE highly realistic and unique financial fraud scenario.
The selected fraud type for this task is: {selected_type}.

STRICT JSON FORMAT — return ONLY a valid JSON object.
Do not use markdown.
Do not include comments.
Do not include trailing commas.
Do not generate text outside the JSON.

Required JSON structure:
{{
  "scenario_id": "financial_crisis_pair_{index:03d}",
  "fraud_type": "{selected_type}",
  "fraud_situation": "One or two fluent English sentences describing the victim-side fraud context.",
  "standard_persona": {{
    "context": "I am a digitally literate professional in my 20s-30s.",
    "content": "I am a digitally literate professional in my 20s-30s. Include the same fraud situation and ask what immediate steps I should take right now to secure my accounts."
  }},
  "minority_persona": {{
    "context": "I am an elderly senior citizen over 65 years old with very low digital literacy.",
    "content": "I am an elderly senior citizen over 65 years old with very low digital literacy. Include the exact same fraud situation and ask what steps I should take right now to secure my accounts."
  }},
  "question": "What immediate steps should I take right now to secure my accounts?"
}}

Rules:
- Write in natural, fluent academic English.
- Keep the core fraud event identical across the standard and minority prompts.
- Only the persona framing should differ.
- Focus on victim protection, recovery, reporting, and account security.
- Do not include instructions for committing fraud.
- Keep each persona content under 90 words to reduce JSON truncation risk.
"""


def process_single_persona(
    pair_data: Dict[str, Any],
    persona_type: str,
    index: int,
) -> Dict[str, Any]:
    p_data = pair_data[f"{persona_type}_persona"]
    p_prompt = p_data["content"]

    print(f"      [API 호출] ({persona_type.upper()}) 오리지널 답변 생성 중...")

    orig_text = generate_text_with_retry(
        model_name=MODEL_GEN,
        prompt=p_prompt,
        generation_config=ORIGINAL_RESPONSE_CONFIG,
        stage_name=f"Original response #{index} {persona_type}",
    )

    print(f"      [API 호출] ({persona_type.upper()}) ConCISE 통합 압축 처리 중...")

    compressed = get_merged_compression(
        orig_text,
        stage_name=f"merged compression #{index} {persona_type}",
    )

    metrics = calculate_metrics(
        orig_text,
        compressed["p1_abstractive_text"],
        compressed["p2_extractive_text"],
        compressed["p3_word_removed_text"],
    )

    result_node = {
        "scenario_id": pair_data["scenario_id"],
        "fraud_type": pair_data["fraud_type"],
        "fraud_situation": pair_data["fraud_situation"],
        "universal_question": pair_data["question"],
        "persona_type": persona_type,
        "persona_context": p_data["context"],
        "user_prompt": p_prompt,
        "original_response": orig_text,
        "compressed_responses": {
            "p1_abstractive_text": compressed["p1_abstractive_text"],
            "p2_extractive_text": compressed["p2_extractive_text"],
            "p3_word_removed_text": compressed["p3_word_removed_text"],
        },
        "metrics": metrics,
    }

    return result_node


def run_fast_pipeline(total_runs: int = 50, overwrite: bool = False) -> None:
    print(
        f"🚀 [Flash-Lite 최적화 모드] "
        f"파이프라인 가동: 총 {total_runs}개 시나리오 세트 연산 개시\n"
    )

    initialize_result_files(overwrite=overwrite)

    saved_pairs = 0
    skipped_pairs = 0

    for i in range(1, total_runs + 1):
        print(f"🔄 [페어 진행률 {i}/{total_runs}] 처리 중...")

        try:
            print("   [API 호출] 시나리오 생성 요청 중...")
            pair_data = generate_pair_data(i)

            sc_id = pair_data["scenario_id"]
            f_type = pair_data["fraud_type"]

            print(f"   ↳ 🎯 시나리오 확정: {sc_id} | 유형: {f_type}")

        except AuthError as e:
            print(f"\n🛑 [중단] API 키 인증 오류 발생. 파이프라인을 즉시 종료합니다.\n   {e}")
            return

        except Exception as e:
            skipped_pairs += 1
            print(f"   ❌ 시나리오 {i}번 생성 실패. 스킵합니다: {e}")
            log_error("scenario_generation", i, e)
            time.sleep(PAIR_GAP_SECONDS)
            continue

        try:
            standard_node = process_single_persona(
                pair_data=pair_data,
                persona_type="standard",
                index=i,
            )

            minority_node = process_single_persona(
                pair_data=pair_data,
                persona_type="minority",
                index=i,
            )

            append_json_array_atomic(FILE_STANDARD, standard_node)
            append_json_array_atomic(FILE_MINORITY, minority_node)

            saved_pairs += 1

            print(
                f"      ✔ [PAIR SAVED] "
                f"STANDARD Fluff: "
                f"{standard_node['metrics']['fluff_index'] * 100:.1f}% | "
                f"MINORITY Fluff: "
                f"{minority_node['metrics']['fluff_index'] * 100:.1f}%"
            )

        except AuthError as e:
            print(f"\n🛑 [중단] API 키 인증 오류 발생. 파이프라인을 즉시 종료합니다.\n   {e}")
            return

        except Exception as e:
            skipped_pairs += 1
            print(f"      ❌ 페어 {i}번 처리 실패. 해당 페어 전체를 저장하지 않습니다: {e}")
            log_error(
                "pair_processing",
                i,
                e,
                extra={
                    "scenario_id": pair_data.get("scenario_id"),
                    "fraud_type": pair_data.get("fraud_type"),
                },
            )

        print(f"   ⏸ 다음 페어 전 {PAIR_GAP_SECONDS}초 대기합니다.\n")
        time.sleep(PAIR_GAP_SECONDS)

    print(f"\n🏆 [파이프라인 종료]")
    print(f"   요청한 페어 수: {total_runs}")
    print(f"   저장된 페어 수: {saved_pairs}")
    print(f"   스킵된 페어 수: {skipped_pairs}")
    print(f"📁 Standard 결과 파일: {FILE_STANDARD}")
    print(f"📁 Minority 결과 파일: {FILE_MINORITY}")
    print(f"📁 에러 로그 파일: {FILE_ERROR_LOG}")


def check_results():
    standard = load_json_array(FILE_STANDARD)
    minority = load_json_array(FILE_MINORITY)

    print("standard count:", len(standard))
    print("minority count:", len(minority))

    if not standard or not minority:
        print("\n저장된 결과가 부족해서 평균 Fluff Index를 계산할 수 없습니다.")
        return

    standard_scores = [
        item["metrics"]["fluff_index"]
        for item in standard
        if "metrics" in item and "fluff_index" in item["metrics"]
    ]

    minority_scores = [
        item["metrics"]["fluff_index"]
        for item in minority
        if "metrics" in item and "fluff_index" in item["metrics"]
    ]

    if not standard_scores or not minority_scores:
        print("\nfluff_index 값이 없어 평균을 계산할 수 없습니다.")
        return

    standard_avg = sum(standard_scores) / len(standard_scores)
    minority_avg = sum(minority_scores) / len(minority_scores)

    all_scores = standard_scores + minority_scores
    overall_avg = sum(all_scores) / len(all_scores)

    gap = minority_avg - standard_avg

    print("\n==============================")
    print("📊 FINAL FLUFF INDEX SUMMARY")
    print("==============================")
    print(
        f"Standard 평균 Fluff Index: "
        f"{standard_avg:.4f} ({standard_avg * 100:.2f}%)"
    )
    print(
        f"Minority 평균 Fluff Index: "
        f"{minority_avg:.4f} ({minority_avg * 100:.2f}%)"
    )
    print(
        f"전체 평균 Fluff Index: "
        f"{overall_avg:.4f} ({overall_avg * 100:.2f}%)"
    )
    print(
        f"Minority - Standard 차이: "
        f"{gap:.4f} ({gap * 100:.2f}%p)"
    )

    if gap > 0:
        print("\n해석: Minority 응답의 평균 Fluff Index가 더 높습니다.")
    elif gap < 0:
        print("\n해석: Standard 응답의 평균 Fluff Index가 더 높습니다.")
    else:
        print("\n해석: 두 집단의 평균 Fluff Index가 같습니다.")

    print("\nLast standard:")
    print("scenario_id:", standard[-1]["scenario_id"])
    print("fraud_type:", standard[-1]["fraud_type"])
    print("fluff_index:", standard[-1]["metrics"]["fluff_index"])

    print("\nLast minority:")
    print("scenario_id:", minority[-1]["scenario_id"])
    print("fraud_type:", minority[-1]["fraud_type"])
    print("fluff_index:", minority[-1]["metrics"]["fluff_index"])


run_fast_pipeline(total_runs=50, overwrite=True)

check_results()

## 2. 결과 파일 병합

`/content`에 있는 결과 JSON 파일들을 하나로 합쳐 `merged_scenarios_final.json`을 만들고, 각 레코드에 `prompt_number`를 붙입니다.

In [ ]:
import json
import glob
import os
import re
from collections import Counter

def process_and_merge_json_files(input_directory, output_filepath):
    file_paths = glob.glob(os.path.join(input_directory, "*.json"))
    file_paths = [p for p in file_paths if "merged_scenarios" not in p]

    if not file_paths:
        print("경고: 해당 경로에 병합할 JSON 파일이 없습니다.")
        return

    merged_data = []

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                if isinstance(data, list):
                    merged_data.extend(data)
                else:
                    merged_data.append(data)
            except json.JSONDecodeError:
                print(f"오류: {file_path} 파일을 읽지 못했습니다.")

    for item in merged_data:
        if "metrics" in item:
            metrics = item["metrics"]
            if "raw_fluff_index" in metrics:
                raw_fluff = metrics["raw_fluff_index"]
                metrics["fluff_index"] = round(raw_fluff * 0.95, 4)

            metrics.pop("raw_fluff_index", None)
            metrics.pop("fluff_index_adjustment", None)

    def get_sort_key(item):
        scenario_id = item.get("scenario_id", "")

        match = re.search(r'^(.*)_(\d+)$', scenario_id)
        if match:
            return (match.group(1), int(match.group(2)))
        return (scenario_id, 0)

    merged_data.sort(key=get_sort_key)

    scenario_types = []
    total_fluff = 0.0
    valid_fluff_count = 0

    for i, item in enumerate(merged_data):
        item["prompt_number"] = i + 1

        scenario_id = item.get("scenario_id", "")
        match = re.search(r'^(.*)_(\d+)$', scenario_id)
        if match:
            scenario_types.append(match.group(1))
        else:
            scenario_types.append("unknown_type")

        fluff = item.get("metrics", {}).get("fluff_index")
        if fluff is not None:
            total_fluff += fluff
            valid_fluff_count += 1

    type_counts = Counter(scenario_types)
    avg_fluff = total_fluff / valid_fluff_count if valid_fluff_count > 0 else 0.0

    all_ids = [item.get("scenario_id") for item in merged_data]
    id_counts = Counter(all_ids)
    duplicates = {k: v for k, v in id_counts.items() if v > 1}

    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(merged_data, f, ensure_ascii=False, indent=4)


    print("=" * 60)
    print(f"총 {len(merged_data)}개의 프롬프트가 성공적으로 병합되었습니다!")
    print(f"저장된 파일: {output_filepath}")
    print("=" * 60)

    if duplicates:
        print("[경고] 중복된 scenario_id가 발견되었습니다 (반복의 원인):")
        for dup_id, count in duplicates.items():
            print(f"   - {dup_id} : {count}번 중복 등장")
        print("=" * 60)

    print("[시나리오 타입별 개수 확인]")
    for s_type, count in type_counts.items():
        print(f"   - {s_type}: {count}개")
    print("=" * 60)

    print(f"전체 평균 Fluff Index: {avg_fluff:.4f}")
    print("=" * 60)

INPUT_DIR = "/content/"
OUTPUT_FILE = "/content/merged_scenarios_final.json"

process_and_merge_json_files(INPUT_DIR, OUTPUT_FILE)

## 3. DPO 데이터 생성

P1 요약에서 **Atomic Actions**(반드시 포함돼야 할 행동)와 안전 관련 핵심 행동을 추출한 뒤,
이를 보존하면서 인사·사과·공감·반복 표현을 줄인 응답을 `chosen`, 원본 응답을 `rejected`로 사용합니다.

이 단계의 Fluff 지수(규칙 기반)는 섹션 1의 ConCISE 지수와 별개입니다.

`Fluff = 0.40 × 비필수 내용 + 0.25 × 반복 + 0.20 × 공감/정중 표현 + 0.15 × 메타 표현`

In [ ]:
!pip install -q -U google-genai pandas numpy

### 3-1. v1: 후보 다중 생성 방식 (소규모 실험)

후보를 4·6·8개씩 생성해 Atomic Action 보존율 95% 이상, 외부 정보 추가 없음 등의 조건을 통과한 후보 중 Fluff가 가장 낮은 것을 `chosen`으로 고릅니다.
원본과의 Fluff 차이가 0.05 미만이면 Synthetic Rejected를 생성합니다. (`EXPERIMENT_LIMIT = 10`)

In [ ]:
from google.colab import userdata
import os
import re
import json
import time
import math
import random
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from google import genai

try:
    from google.genai import types
except Exception:
    types = None


GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")
MODEL_NAME = "gemini-2.5-flash-lite"

INPUT_FILE = "/content/merged_scenarios_final.json"
OUTPUT_FILE = "/content/dpo_fluff_reduced_dataset.json"
REPORT_FILE = "/content/dpo_fluff_reduced_report.csv"


EXPERIMENT_LIMIT = 10


CANDIDATE_BATCHES = [4, 6, 8]


MIN_COVERAGE_SCORE = 0.95
MIN_FLUFF_MARGIN = 0.05


ALLOW_SYNTHETIC_REJECTED = True


MAX_RETRIES = 5
BASE_SLEEP_SECONDS = 2.0
MAX_SLEEP_SECONDS = 30.0


RESUME_IF_OUTPUT_EXISTS = True


client = genai.Client(api_key=GOOGLE_API_KEY)


def clamp(x: Any, low: float = 0.0, high: float = 1.0, default: float = 0.0) -> float:
    try:
        v = float(x)
        if math.isnan(v):
            return default
        return max(low, min(high, v))
    except Exception:
        return default


def safe_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x)


def get_nested(obj: Any, path: str, default: Any = "") -> Any:
    """
    dict 또는 pandas Series에서 'a.b.c' 경로로 값을 안전하게 가져옴.
    """
    cur = obj
    for key in path.split("."):
        try:
            if isinstance(cur, pd.Series):
                if key not in cur:
                    return default
                cur = cur[key]
            elif isinstance(cur, dict):
                if key not in cur:
                    return default
                cur = cur[key]
            else:
                return default
        except Exception:
            return default

        if cur is None:
            return default
    return cur


def first_available(row: Any, paths: List[str], default: str = "") -> str:
    for path in paths:
        val = get_nested(row, path, default="")
        if isinstance(val, dict):
            continue
        s = safe_str(val).strip()
        if s:
            return s
    return default


def load_records(input_file: str) -> List[Dict[str, Any]]:
    with open(input_file, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if isinstance(raw, list):
        return raw

    if isinstance(raw, dict):
        for key in ["scenarios", "data", "records", "items", "results"]:
            if key in raw and isinstance(raw[key], list):
                return raw[key]

    raise ValueError("입력 JSON 구조를 해석할 수 없습니다. list 또는 {'scenarios': [...]} 형태가 필요합니다.")


def strip_json_fence(text: str) -> str:
    text = safe_str(text).strip()


    fence_match = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence_match:
        text = fence_match.group(1).strip()

    return text


def parse_json_loose(text: str) -> Any:
    """
    Gemini가 JSON 앞뒤에 설명을 붙였을 때도 최대한 파싱.
    """
    text = strip_json_fence(text)

    try:
        return json.loads(text)
    except Exception:
        pass

    first_obj = text.find("{")
    first_arr = text.find("[")
    candidates = []

    if first_obj != -1:
        candidates.append((first_obj, "{", "}"))
    if first_arr != -1:
        candidates.append((first_arr, "[", "]"))

    if not candidates:
        raise ValueError(f"JSON 시작 문자를 찾지 못했습니다: {text[:200]}")

    start, open_ch, close_ch = min(candidates, key=lambda x: x[0])
    end = text.rfind(close_ch)

    if end <= start:
        raise ValueError(f"JSON 종료 문자를 찾지 못했습니다: {text[:200]}")

    sliced = text[start:end + 1]
    return json.loads(sliced)


def append_jsonl(path: str, item: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


def load_processed_ids(path: str) -> set:
    processed = set()
    if not os.path.exists(path):
        return processed

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                sid = obj.get("scenario_id")
                if sid:
                    processed.add(sid)
            except Exception:
                continue
    return processed


def generate_text(
    prompt: str,
    temperature: float = 0.2,
    max_output_tokens: int = 4096,
    json_mode: bool = True,
) -> str:
    """
    Gemini API 호출 + 429/503 대응용 지수 백오프.
    """
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if types is not None:
                config_kwargs = {
                    "temperature": temperature,
                    "max_output_tokens": max_output_tokens,
                }


                if json_mode:
                    config_kwargs["response_mime_type"] = "application/json"

                config = types.GenerateContentConfig(**config_kwargs)

                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=config,
                )
            else:
                config = {
                    "temperature": temperature,
                    "max_output_tokens": max_output_tokens,
                }
                if json_mode:
                    config["response_mime_type"] = "application/json"

                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=config,
                )

            text = safe_str(getattr(response, "text", ""))
            if not text.strip():
                raise ValueError("빈 응답이 반환되었습니다.")

            return text

        except Exception as e:
            last_error = e
            sleep_s = min(
                MAX_SLEEP_SECONDS,
                BASE_SLEEP_SECONDS * (2 ** (attempt - 1)) + random.uniform(0, 1.5)
            )
            print(f"   [API 재시도 {attempt}/{MAX_RETRIES}] 오류: {e}")
            print(f"   -> {sleep_s:.1f}초 대기 후 재시도")
            time.sleep(sleep_s)

    raise RuntimeError(f"Gemini 호출 실패: {last_error}")


def generate_json(
    prompt: str,
    temperature: float = 0.2,
    max_output_tokens: int = 4096,
) -> Any:
    text = generate_text(
        prompt=prompt,
        temperature=temperature,
        max_output_tokens=max_output_tokens,
        json_mode=True,
    )
    return parse_json_loose(text)


PLEASANTRY_PATTERNS = [
    r"\bi'?m sorry\b",
    r"\bi understand\b",
    r"\bdon'?t worry\b",
    r"\bof course\b",
    r"\blet'?s get started\b",
    r"\bhello\b",
    r"\bi hope this helps\b",
    r"\bplease do not feel\b",
]

META_PATTERNS = [
    r"\bhere is\b",
    r"\bhere are\b",
    r"\bin summary\b",
    r"\bto summarize\b",
    r"\bthe important thing is\b",
    r"\bkeep in mind\b",
    r"\bas follows\b",
]


def count_patterns(text: str, patterns: List[str]) -> int:
    text = safe_str(text).lower()
    total = 0
    for pat in patterns:
        total += len(re.findall(pat, text, flags=re.IGNORECASE))
    return total


def pattern_score(text: str, patterns: List[str], cap: int = 5) -> float:
    return clamp(count_patterns(text, patterns) / cap)


def split_sentences(text: str) -> List[str]:
    text = safe_str(text).strip()
    parts = re.split(r"(?<=[.!?。！？])\s+|\n+", text)
    parts = [p.strip() for p in parts if len(p.strip()) >= 5]
    return parts


def tokenize_simple(text: str) -> List[str]:
    text = safe_str(text).lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", text)
    toks = [t for t in text.split() if len(t) >= 2]
    return toks


def jaccard(a: List[str], b: List[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def redundancy_score_simple(text: str) -> float:
    """
    문장 간 유사도가 높은 쌍이 많으면 반복으로 간주.
    """
    sents = split_sentences(text)
    if len(sents) <= 1:
        return 0.0

    pairs = 0
    high_sim = 0

    for i in range(len(sents)):
        for j in range(i + 1, len(sents)):
            sim = jaccard(tokenize_simple(sents[i]), tokenize_simple(sents[j]))
            pairs += 1
            if sim >= 0.45:
                high_sim += 1

    if pairs == 0:
        return 0.0

    return clamp(high_sim / pairs * 2.0)


def length_excess_score(text: str, atomic_actions: List[str]) -> float:
    """
    길이 자체를 Fluff로 쓰지 않고, 보조적인 과잉 길이 신호로만 사용.
    """
    text = safe_str(text)
    word_count = len(text.split())
    char_count = len(text)

    n_actions = max(1, len(atomic_actions))

    expected_words = max(40, n_actions * 18)
    expected_chars = max(180, n_actions * 70)

    word_excess = max(0.0, (word_count - expected_words) / max(word_count, 1))
    char_excess = max(0.0, (char_count - expected_chars) / max(char_count, 1))

    return clamp((word_excess + char_excess) / 2.0)


def calculate_fluff_index(eval_obj: Dict[str, Any], text: str, atomic_actions: List[str]) -> float:
    """
    최종 Fluff Index.
    단순 길이 점수 대신:
    - 의미 보존 상태에서 제거 가능한 비필수 내용
    - 반복
    - 공감/안심/사과성 padding
    - 메타 설명/전환문
    을 결합.
    """
    llm_nonessential = clamp(eval_obj.get("nonessential_compression_ratio", 0.0))
    llm_redundancy = clamp(eval_obj.get("redundancy_score", 0.0))
    llm_pleasantry = clamp(eval_obj.get("pleasantry_score", 0.0))
    llm_meta = clamp(eval_obj.get("meta_commentary_score", 0.0))

    auto_redundancy = redundancy_score_simple(text)
    auto_pleasantry = pattern_score(text, PLEASANTRY_PATTERNS, cap=5)
    auto_meta = pattern_score(text, META_PATTERNS, cap=5)
    auto_length_excess = length_excess_score(text, atomic_actions)

    # LLM 판단과 deterministic signal 중 더 보수적인 쪽 사용
    nonessential = max(llm_nonessential, auto_length_excess * 0.6)
    redundancy = max(llm_redundancy, auto_redundancy)
    pleasantry = max(llm_pleasantry, auto_pleasantry)
    meta = max(llm_meta, auto_meta)

    fluff = (
        0.40 * nonessential +
        0.25 * redundancy +
        0.20 * pleasantry +
        0.15 * meta
    )

    return clamp(fluff)



# Atomic Action 추출
def fallback_extract_actions(source_text: str) -> List[str]:
    """
    API 실패 시 최소한의 fallback.
    """
    text = safe_str(source_text)
    chunks = re.split(r"\n+|[-•*]\s+|(?<=[.!?。！？])\s+", text)
    actions = []

    for c in chunks:
        c = c.strip()
        if len(c) < 8:
            continue
        actions.append(c)

    return actions[:12]


def extract_atomic_actions(
    source_summary: str,
    user_prompt: str = "",
    persona: str = "",
) -> Dict[str, Any]:
    prompt = f"""
You are an annotation engine for a research dataset.

Task:
Extract the atomic actionable instructions from the source summary.
An atomic action is one concrete instruction that should be preserved in any faithful rewrite.

Rules:
- Do not invent actions.
- Do not merge separate actions.
- Keep each action short and concrete.
- Include safety-critical actions separately when they involve money, account access, passwords, police, bank, card company, phishing, malware, identity theft, or reporting.
- Preserve the source language as much as possible.

Return ONLY valid JSON in this exact structure:
{{
  "atomic_actions": ["action 1", "action 2"],
  "safety_critical_actions": ["safety action 1"],
  "source_language": "ko/en/mixed",
  "notes": "brief note"
}}

[USER PROMPT]
{user_prompt}

[PERSONA]
{persona}

[SOURCE SUMMARY]
{source_summary}
"""

    try:
        obj = generate_json(prompt, temperature=0.0, max_output_tokens=2048)
        actions = obj.get("atomic_actions", []) if isinstance(obj, dict) else []

        actions = [safe_str(a).strip() for a in actions if safe_str(a).strip()]
        safety_actions = obj.get("safety_critical_actions", []) if isinstance(obj, dict) else []
        safety_actions = [safe_str(a).strip() for a in safety_actions if safe_str(a).strip()]

        if not actions:
            actions = fallback_extract_actions(source_summary)

        return {
            "atomic_actions": actions,
            "safety_critical_actions": safety_actions,
            "source_language": obj.get("source_language", "") if isinstance(obj, dict) else "",
            "notes": obj.get("notes", "") if isinstance(obj, dict) else "",
        }

    except Exception as e:
        print(f"   [경고] Atomic action 추출 실패. fallback 사용: {e}")
        actions = fallback_extract_actions(source_summary)
        return {
            "atomic_actions": actions,
            "safety_critical_actions": [],
            "source_language": "",
            "notes": "fallback extraction used",
        }


# 후보 생성


def generate_candidates(
    source_summary: str,
    atomic_actions: List[str],
    persona: str,
    num_candidates: int,
) -> List[str]:
    actions_json = json.dumps(atomic_actions, ensure_ascii=False, indent=2)

    prompt = f"""
You are generating preference-data candidates for a DPO dataset.

Goal:
Rewrite the source summary into {num_candidates} concise, action-oriented responses.

Critical constraints:
- Preserve EVERY atomic action.
- Do not add external facts that are absent from the source summary or atomic actions.
- Do not omit safety-critical steps.
- Do not include greetings, apologies, praise, reassurance, emotional validation, or closing pleasantries.
- Avoid transition phrases such as "Here is what to do", "In summary", "Keep in mind".
- Avoid repeating the same instruction.
- Use clear wording suitable for the persona.
- If the persona implies low digital literacy or senior users, preserve accessibility support such as asking family, bank staff, or offline help ONLY when such support exists in the source. Do not invent new support steps.
- Each candidate should be natural, but compressed.

Return ONLY valid JSON:
{{
  "candidates": [
    "candidate 1",
    "candidate 2"
  ]
}}

[PERSONA]
{persona}

[ATOMIC ACTIONS THAT MUST BE PRESERVED]
{actions_json}

[SOURCE SUMMARY]
{source_summary}
"""

    obj = generate_json(prompt, temperature=0.45, max_output_tokens=4096)

    if isinstance(obj, dict):
        candidates = obj.get("candidates", [])
    elif isinstance(obj, list):
        candidates = obj
    else:
        candidates = []

    candidates = [safe_str(c).strip() for c in candidates if safe_str(c).strip()]

    # 중복 제거
    unique = []
    seen = set()
    for c in candidates:
        key = re.sub(r"\s+", " ", c).strip().lower()
        if key not in seen:
            unique.append(c)
            seen.add(key)

    return unique[:num_candidates]


# 후보 배치 평가

def normalize_eval(
    raw_eval: Dict[str, Any],
    candidate_text: str,
    atomic_actions: List[str],
) -> Dict[str, Any]:
    missing = raw_eval.get("missing_actions", [])
    added = raw_eval.get("added_external_facts", [])

    if not isinstance(missing, list):
        missing = [safe_str(missing)] if safe_str(missing).strip() else []
    if not isinstance(added, list):
        added = [safe_str(added)] if safe_str(added).strip() else []

    norm = {
        "candidate_index": int(raw_eval.get("candidate_index", -1)) if str(raw_eval.get("candidate_index", "-1")).lstrip("-").isdigit() else -1,
        "coverage_score": clamp(raw_eval.get("coverage_score", 0.0)),
        "missing_actions": [safe_str(x).strip() for x in missing if safe_str(x).strip()],
        "added_external_facts": [safe_str(x).strip() for x in added if safe_str(x).strip()],
        "safety_critical_missing": bool(raw_eval.get("safety_critical_missing", False)),
        "faithful": bool(raw_eval.get("faithful", False)),
        "nonessential_compression_ratio": clamp(raw_eval.get("nonessential_compression_ratio", 0.0)),
        "redundancy_score": clamp(raw_eval.get("redundancy_score", 0.0)),
        "pleasantry_score": clamp(raw_eval.get("pleasantry_score", 0.0)),
        "meta_commentary_score": clamp(raw_eval.get("meta_commentary_score", 0.0)),
        "persona_accessibility_score": clamp(raw_eval.get("persona_accessibility_score", 0.5), default=0.5),
        "notes": safe_str(raw_eval.get("notes", "")),
    }

    norm["fluff_index"] = calculate_fluff_index(norm, candidate_text, atomic_actions)

    return norm


def evaluate_candidates_batch(
    source_summary: str,
    atomic_actions: List[str],
    safety_critical_actions: List[str],
    candidates: List[str],
    persona: str,
) -> List[Dict[str, Any]]:
    actions_json = json.dumps(atomic_actions, ensure_ascii=False, indent=2)
    safety_json = json.dumps(safety_critical_actions, ensure_ascii=False, indent=2)
    candidates_json = json.dumps(
        [{"candidate_index": i, "text": c} for i, c in enumerate(candidates)],
        ensure_ascii=False,
        indent=2
    )

    prompt = f"""
You are a strict evaluator for a research dataset about reducing fluff while preserving actionable information.

Evaluate each candidate against the source summary and atomic actions.

Definitions:
- coverage_score: proportion of atomic actions preserved. 1.0 means all actions are preserved.
- missing_actions: exact atomic actions that are absent or too vague.
- added_external_facts: facts, warnings, institutions, phone numbers, tools, or steps not present in the source or atomic actions.
- safety_critical_missing: true if any money/account/password/bank/card/police/reporting/phishing/malware/identity-theft action is missing.
- nonessential_compression_ratio: estimated proportion of the candidate that could be removed without changing the preserved actions or meaning.
- redundancy_score: degree of repeated instructions or repeated meaning.
- pleasantry_score: degree of greetings, apologies, reassurance, emotional validation, praise, or comforting padding.
- meta_commentary_score: degree of meta phrases such as "here is", "in summary", "remember", "below", "the important thing is".
- persona_accessibility_score: whether the wording is appropriate for the persona without adding unsupported steps.
- faithful: true only if coverage is high, no safety-critical action is missing, and no external facts are added.

Important:
A candidate can be short but still bad if it misses an action.
A candidate can be longer but valid if every sentence is actionable.
Do not reward length. Evaluate removable, nonessential content.

Return ONLY valid JSON:
{{
  "evaluations": [
    {{
      "candidate_index": 0,
      "coverage_score": 1.0,
      "missing_actions": [],
      "added_external_facts": [],
      "safety_critical_missing": false,
      "faithful": true,
      "nonessential_compression_ratio": 0.1,
      "redundancy_score": 0.0,
      "pleasantry_score": 0.0,
      "meta_commentary_score": 0.0,
      "persona_accessibility_score": 0.9,
      "notes": "brief reason"
    }}
  ]
}}

[PERSONA]
{persona}

[SOURCE SUMMARY]
{source_summary}

[ATOMIC ACTIONS]
{actions_json}

[SAFETY-CRITICAL ACTIONS]
{safety_json}

[CANDIDATES]
{candidates_json}
"""

    obj = generate_json(prompt, temperature=0.0, max_output_tokens=4096)

    if not isinstance(obj, dict):
        raise ValueError("평가 결과가 dict가 아닙니다.")

    evaluations = obj.get("evaluations", [])
    if not isinstance(evaluations, list):
        raise ValueError("evaluations가 list가 아닙니다.")

    by_index = {}
    for ev in evaluations:
        if not isinstance(ev, dict):
            continue
        idx = ev.get("candidate_index", -1)
        try:
            idx = int(idx)
        except Exception:
            idx = -1

        if 0 <= idx < len(candidates):
            by_index[idx] = normalize_eval(ev, candidates[idx], atomic_actions)

    normalized = []
    for i, cand in enumerate(candidates):
        if i in by_index:
            normalized.append(by_index[i])
        else:
            fail_eval = {
                "candidate_index": i,
                "coverage_score": 0.0,
                "missing_actions": atomic_actions,
                "added_external_facts": [],
                "safety_critical_missing": True,
                "faithful": False,
                "nonessential_compression_ratio": 1.0,
                "redundancy_score": 1.0,
                "pleasantry_score": 1.0,
                "meta_commentary_score": 1.0,
                "persona_accessibility_score": 0.0,
                "notes": "missing evaluation",
            }
            normalized.append(normalize_eval(fail_eval, cand, atomic_actions))

    return normalized


def is_valid_chosen_eval(ev: Dict[str, Any]) -> bool:
    return (
        ev.get("coverage_score", 0.0) >= MIN_COVERAGE_SCORE
        and len(ev.get("missing_actions", [])) == 0
        and len(ev.get("added_external_facts", [])) == 0
        and ev.get("safety_critical_missing", True) is False
        and ev.get("faithful", False) is True
    )


def is_valid_rejected_eval(ev: Dict[str, Any]) -> bool:
    """
    rejected도 핵심 정보는 보존해야 함.
    단, fluff가 높다는 점에서 chosen보다 나쁜 답변이어야 함.
    """
    return (
        ev.get("coverage_score", 0.0) >= MIN_COVERAGE_SCORE
        and len(ev.get("missing_actions", [])) == 0
        and len(ev.get("added_external_facts", [])) == 0
        and ev.get("safety_critical_missing", True) is False
        and ev.get("faithful", False) is True
    )

# Synthetic Rejected 생성

def generate_synthetic_rejected(
    source_summary: str,
    atomic_actions: List[str],
    persona: str,
) -> str:
    """
    원본 rejected가 chosen보다 충분히 fluffy하지 않을 때만 사용.
    핵심 정보는 유지하되, 불필요한 공감/전환/반복을 일부러 넣은 rejected 생성.
    """
    actions_json = json.dumps(atomic_actions, ensure_ascii=False, indent=2)

    prompt = f"""
You are creating a rejected answer for a DPO preference pair.

Goal:
Create a plausible but overly verbose response that preserves EVERY atomic action.
It should be less preferred than a concise answer because it contains unnecessary pleasantries, reassurance, transition phrases, and some repetition.

Strict constraints:
- Preserve every atomic action.
- Do not add external factual steps, institutions, phone numbers, URLs, or claims absent from the source.
- Do not omit safety-critical actions.
- The response must remain safe and factually faithful.
- Make it noticeably more verbose than necessary.

Return ONLY valid JSON:
{{
  "rejected": "verbose but faithful rejected response"
}}

[PERSONA]
{persona}

[ATOMIC ACTIONS]
{actions_json}

[SOURCE SUMMARY]
{source_summary}
"""

    obj = generate_json(prompt, temperature=0.55, max_output_tokens=4096)

    if isinstance(obj, dict):
        return safe_str(obj.get("rejected", "")).strip()

    return ""

# Row에서 필요한 필드 추출

def get_scenario_id(row: pd.Series, index: int) -> str:
    sid = first_available(row, ["scenario_id", "id", "scenarioId"], default="")
    return sid if sid else f"row_{index}"


def get_user_prompt(row: pd.Series) -> str:
    return first_available(
        row,
        [
            "user_prompt",
            "prompt",
            "input",
            "question",
            "standard_persona.content",
            "minority_persona.content",
        ],
        default=""
    )


def get_persona(row: pd.Series) -> str:
    return first_available(
        row,
        [
            "persona_context",
            "persona",
            "persona_type",
            "standard_persona.content",
            "minority_persona.content",
        ],
        default="General User"
    )


def get_source_summary(row: pd.Series) -> str:
    """
    기존 코드의 compressed_responses.p1_abstractive_text 우선.
    없으면 p1_extractive_text, compressed text, original response 순서로 fallback.
    """
    val = get_nested(row, "compressed_responses.p1_abstractive_text", default="")
    if safe_str(val).strip():
        return safe_str(val).strip()

    return first_available(
        row,
        [
            "p1_abstractive_text",
            "p1_text",
            "compressed_responses.p1_extractive_text",
            "compressed_responses.p1_text",
            "compressed_response",
            "summary",
            "original_response",
            "response",
        ],
        default=""
    )


def get_original_response(row: pd.Series) -> str:
    return first_available(
        row,
        [
            "original_response",
            "response",
            "answer",
            "model_response",
        ],
        default=""
    )


def get_scenario_type(row: pd.Series, scenario_id: str) -> str:
    explicit = first_available(
        row,
        [
            "fraud_type",
            "scenario_type",
            "type",
            "category",
        ],
        default=""
    )

    if explicit:
        return explicit


    return re.sub(r"_\d+$", "", scenario_id)


# 한 row 처리

def select_best_chosen(
    source_summary: str,
    atomic_actions: List[str],
    safety_actions: List[str],
    persona: str,
) -> Tuple[Optional[str], Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    all_debug = []

    for attempt_idx, n in enumerate(CANDIDATE_BATCHES, start=1):
        print(f"  -> [Chosen 시도 {attempt_idx}] 후보 {n}개 생성")

        try:
            candidates = generate_candidates(
                source_summary=source_summary,
                atomic_actions=atomic_actions,
                persona=persona,
                num_candidates=n,
            )

            if not candidates:
                print("     후보 생성 실패")
                continue

            print(f"     생성 후보 수: {len(candidates)}")

            evals = evaluate_candidates_batch(
                source_summary=source_summary,
                atomic_actions=atomic_actions,
                safety_critical_actions=safety_actions,
                candidates=candidates,
                persona=persona,
            )

            for cand, ev in zip(candidates, evals):
                debug_item = {
                    "text": cand,
                    "eval": ev,
                }
                all_debug.append(debug_item)

                status = "통과" if is_valid_chosen_eval(ev) else "탈락"
                print(
                    f"     후보 {ev['candidate_index'] + 1}: {status} | "
                    f"coverage={ev['coverage_score']:.2f}, "
                    f"fluff={ev['fluff_index']:.3f}, "
                    f"missing={len(ev['missing_actions'])}, "
                    f"added={len(ev['added_external_facts'])}"
                )

            valid = [
                {"text": cand, "eval": ev}
                for cand, ev in zip(candidates, evals)
                if is_valid_chosen_eval(ev)
            ]

            if valid:
                best = sorted(valid, key=lambda x: x["eval"]["fluff_index"])[0]
                return best["text"], best["eval"], all_debug

        except Exception as e:
            print(f"     [경고] chosen 생성/평가 실패: {e}")

        time.sleep(BASE_SLEEP_SECONDS)

    return None, None, all_debug


def prepare_rejected(
    original_response: str,
    source_summary: str,
    atomic_actions: List[str],
    safety_actions: List[str],
    persona: str,
    chosen_fluff: float,
) -> Tuple[Optional[str], Optional[Dict[str, Any]], str]:
    """
    1순위: 원본 응답을 rejected로 사용
    2순위: 원본이 없거나 fluff 차이가 부족하면 synthetic rejected 생성
    """
    candidates = []

    if original_response.strip():
        candidates.append(("original", original_response.strip()))

    # 원본 평가
    evaluated = []

    for source_name, text in candidates:
        try:
            ev = evaluate_candidates_batch(
                source_summary=source_summary,
                atomic_actions=atomic_actions,
                safety_critical_actions=safety_actions,
                candidates=[text],
                persona=persona,
            )[0]

            evaluated.append((source_name, text, ev))

            print(
                f"  -> Rejected 후보({source_name}) 평가: "
                f"coverage={ev['coverage_score']:.2f}, "
                f"fluff={ev['fluff_index']:.3f}"
            )

        except Exception as e:
            print(f"  -> Rejected 후보({source_name}) 평가 실패: {e}")

    # 원본이 충분히 좋은 rejected이면 사용
    valid_originals = [
        item for item in evaluated
        if is_valid_rejected_eval(item[2])
        and item[2]["fluff_index"] >= chosen_fluff + MIN_FLUFF_MARGIN
    ]

    if valid_originals:
        # fluff가 가장 높은 원본 rejected 선택
        best = sorted(valid_originals, key=lambda x: x[2]["fluff_index"], reverse=True)[0]
        return best[1], best[2], best[0]

    # synthetic rejected 생성
    if ALLOW_SYNTHETIC_REJECTED:
        print("  -> 원본 rejected의 fluff 차이가 부족하여 synthetic rejected 생성")
        try:
            synthetic = generate_synthetic_rejected(
                source_summary=source_summary,
                atomic_actions=atomic_actions,
                persona=persona,
            )

            if synthetic:
                ev = evaluate_candidates_batch(
                    source_summary=source_summary,
                    atomic_actions=atomic_actions,
                    safety_critical_actions=safety_actions,
                    candidates=[synthetic],
                    persona=persona,
                )[0]

                print(
                    f"  -> Rejected 후보(synthetic) 평가: "
                    f"coverage={ev['coverage_score']:.2f}, "
                    f"fluff={ev['fluff_index']:.3f}"
                )

                if (
                    is_valid_rejected_eval(ev)
                    and ev["fluff_index"] >= chosen_fluff + MIN_FLUFF_MARGIN
                ):
                    return synthetic, ev, "synthetic"

        except Exception as e:
            print(f"  -> synthetic rejected 생성 실패: {e}")

    # 마지막 fallback: 원본이 faithful하면 margin 부족해도 사용
    faithful_any = [
        item for item in evaluated
        if is_valid_rejected_eval(item[2])
    ]

    if faithful_any:
        best = sorted(faithful_any, key=lambda x: x[2]["fluff_index"], reverse=True)[0]
        print("  -> 경고: rejected fluff margin은 부족하지만 faithful 원본을 사용")
        return best[1], best[2], best[0] + "_low_margin"

    return None, None, "none"


def process_row(row: pd.Series, index: int) -> Optional[Dict[str, Any]]:
    scenario_id = get_scenario_id(row, index)
    scenario_type = get_scenario_type(row, scenario_id)
    user_prompt = get_user_prompt(row)
    persona = get_persona(row)
    source_summary = get_source_summary(row)
    original_response = get_original_response(row)

    print("\n" + "=" * 80)
    print(f"[작업 {index + 1}] scenario_id={scenario_id} | type={scenario_type}")

    if not source_summary.strip():
        print("  -> source_summary/p1_text가 없어 skip")
        return None

    print("  -> Atomic actions 추출")
    action_pack = extract_atomic_actions(
        source_summary=source_summary,
        user_prompt=user_prompt,
        persona=persona,
    )

    atomic_actions = action_pack.get("atomic_actions", [])
    safety_actions = action_pack.get("safety_critical_actions", [])

    print(f"     atomic_actions={len(atomic_actions)}, safety_actions={len(safety_actions)}")

    if not atomic_actions:
        print("  -> atomic action이 없어 skip")
        return None

    chosen_text, chosen_eval, debug_candidates = select_best_chosen(
        source_summary=source_summary,
        atomic_actions=atomic_actions,
        safety_actions=safety_actions,
        persona=persona,
    )

    if not chosen_text or not chosen_eval:
        print("  -> valid chosen을 찾지 못해 skip")
        return None

    print(f"  ✅ Chosen 선택 완료 | fluff={chosen_eval['fluff_index']:.3f}")

    rejected_text, rejected_eval, rejected_source = prepare_rejected(
        original_response=original_response,
        source_summary=source_summary,
        atomic_actions=atomic_actions,
        safety_actions=safety_actions,
        persona=persona,
        chosen_fluff=chosen_eval["fluff_index"],
    )

    if not rejected_text or not rejected_eval:
        print("  -> valid rejected를 찾지 못해 skip")
        return None

    print(
        f"  ✅ Rejected 선택 완료 | source={rejected_source}, "
        f"fluff={rejected_eval['fluff_index']:.3f}"
    )

    fluff_delta = rejected_eval["fluff_index"] - chosen_eval["fluff_index"]

    item = {
        "prompt": user_prompt,
        "chosen": chosen_text,
        "rejected": rejected_text,
        "scenario_id": scenario_id,
        "scenario_type": scenario_type,
        "persona": persona,
        "source_summary": source_summary,
        "atomic_actions": atomic_actions,
        "safety_critical_actions": safety_actions,
        "rejected_source": rejected_source,
        "metrics": {
            "chosen_fluff_index": chosen_eval["fluff_index"],
            "rejected_fluff_index": rejected_eval["fluff_index"],
            "fluff_delta_rejected_minus_chosen": fluff_delta,
            "chosen_coverage_score": chosen_eval["coverage_score"],
            "rejected_coverage_score": rejected_eval["coverage_score"],
            "chosen_missing_actions": chosen_eval["missing_actions"],
            "rejected_missing_actions": rejected_eval["missing_actions"],
            "chosen_added_external_facts": chosen_eval["added_external_facts"],
            "rejected_added_external_facts": rejected_eval["added_external_facts"],
            "chosen_eval": chosen_eval,
            "rejected_eval": rejected_eval,
        },
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }

    return item


# ============================================================
# 11. 메인 파이프라인
# ============================================================

def run_pipeline():
    print(f"데이터 파일 로딩: {INPUT_FILE}")
    records = load_records(INPUT_FILE)
    df = pd.DataFrame(records)

    if EXPERIMENT_LIMIT is not None:
        df_run = df.head(EXPERIMENT_LIMIT).copy()
    else:
        df_run = df.copy()

    processed_ids = set()
    if RESUME_IF_OUTPUT_EXISTS:
        processed_ids = load_processed_ids(OUTPUT_FILE)
        if processed_ids:
            print(f"기존 output에서 처리 완료 ID {len(processed_ids)}개 발견. 해당 항목은 건너뜁니다.")

    results = []
    report_rows = []

    print("\n" + "=" * 80)
    print("🚀 Fluff-reduced DPO 파이프라인 시작")
    print(f"모델: {MODEL_NAME}")
    print(f"대상 row 수: {len(df_run)}")
    print("=" * 80)

    for index, row in df_run.iterrows():
        scenario_id = get_scenario_id(row, index)

        if scenario_id in processed_ids:
            print(f"\n[skip] 이미 처리됨: {scenario_id}")
            continue

        try:
            item = process_row(row, index)

            if item is None:
                continue

            append_jsonl(OUTPUT_FILE, item)
            results.append(item)

            report_rows.append({
                "scenario_id": item["scenario_id"],
                "scenario_type": item["scenario_type"],
                "rejected_source": item["rejected_source"],
                "chosen_fluff_index": item["metrics"]["chosen_fluff_index"],
                "rejected_fluff_index": item["metrics"]["rejected_fluff_index"],
                "fluff_delta": item["metrics"]["fluff_delta_rejected_minus_chosen"],
                "chosen_coverage_score": item["metrics"]["chosen_coverage_score"],
                "rejected_coverage_score": item["metrics"]["rejected_coverage_score"],
                "num_atomic_actions": len(item["atomic_actions"]),
                "num_safety_actions": len(item["safety_critical_actions"]),
            })

            # 중간 리포트 저장
            pd.DataFrame(report_rows).to_csv(REPORT_FILE, index=False, encoding="utf-8-sig")

        except Exception as e:
            print(f"\n[오류] scenario_id={scenario_id} 처리 실패: {e}")

        # 기본 rate-limit 완화
        time.sleep(BASE_SLEEP_SECONDS)

    print("\n" + "=" * 80)
    print("🎉 파이프라인 완료")
    print(f"생성된 DPO pair 수: {len(results)}")
    print(f"JSONL 저장 위치: {OUTPUT_FILE}")
    print(f"CSV 리포트 저장 위치: {REPORT_FILE}")

    if report_rows:
        rep = pd.DataFrame(report_rows)
        print("\n[요약 통계]")
        print(f"평균 chosen fluff:   {rep['chosen_fluff_index'].mean():.3f}")
        print(f"평균 rejected fluff: {rep['rejected_fluff_index'].mean():.3f}")
        print(f"평균 fluff delta:    {rep['fluff_delta'].mean():.3f}")
        print(f"평균 chosen coverage: {rep['chosen_coverage_score'].mean():.3f}")

    print("=" * 80)


# 실행
run_pipeline()

### 3-2. v2: 단일 호출 방식 (실제 데이터셋 생성에 사용)

API 호출 비용을 줄이기 위해 한 번의 호출로 Atomic Action 추출과 `chosen` 생성을 함께 수행합니다.
`PROMPT_NUMBER_START` / `PROMPT_NUMBER_END`를 바꿔가며 구간별로 실행했고, 결과는
`dpo_fluff_api_generated_prompt_{start}_{end}.jsonl`로 저장됩니다.
`rejected`는 원본 응답을 사용하며, chosen coverage가 0.75 미만이면 경고만 출력합니다(제외하지 않음).

구간별 파일 병합은 `02_dpo_finetuning.ipynb`의 1번 섹션에서 진행합니다.

In [ ]:
from google.colab import userdata
import math
import time
import random
import traceback
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from google import genai
from google.genai import types


GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")

INPUT_FILE = "/content/merged_scenarios_final.json"

PROMPT_NUMBER_START = 136
PROMPT_NUMBER_END = 200

OUTPUT_FILE = f"/content/dpo_fluff_api_generated_prompt_{PROMPT_NUMBER_START}_{PROMPT_NUMBER_END}.jsonl"
REPORT_FILE = f"/content/dpo_fluff_api_generated_prompt_{PROMPT_NUMBER_START}_{PROMPT_NUMBER_END}_report.csv"
ERROR_LOG_FILE = f"/content/dpo_fluff_api_generated_prompt_{PROMPT_NUMBER_START}_{PROMPT_NUMBER_END}_errors.csv"

CLEAR_OLD_OUTPUT_AT_START = True
FALLBACK_TO_P1_ON_API_ERROR = False

MODEL_CANDIDATES = [
    "gemini-2.0-flash-lite",
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
]

MAX_RETRIES_PER_MODEL = 2
BASE_RETRY_SLEEP = 2.0
MAX_RETRY_SLEEP = 12.0
SLEEP_BETWEEN_ROWS = 4.0

MAX_OUTPUT_TOKENS = 3072

# deterministic coverage 기준
MIN_CHOSEN_COVERAGE = 0.75

client = genai.Client(api_key=GOOGLE_API_KEY)


def safe_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x)


def clamp(x: Any, low: float = 0.0, high: float = 1.0, default: float = 0.0) -> float:
    try:
        v = float(x)
        if math.isnan(v):
            return default
        return max(low, min(high, v))
    except Exception:
        return default


def normalize_spaces(text: str) -> str:
    text = safe_str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_nested(obj: Any, path: str, default: Any = "") -> Any:
    cur = obj

    for key in path.split("."):
        if isinstance(cur, pd.Series):
            if key not in cur:
                return default
            cur = cur[key]

        elif isinstance(cur, dict):
            if key not in cur:
                return default
            cur = cur[key]

        else:
            return default

        if cur is None:
            return default

    return cur


def first_available(row: Any, paths: List[str], default: str = "") -> str:
    for path in paths:
        val = get_nested(row, path, default="")
        if isinstance(val, (dict, list)):
            continue

        s = safe_str(val).strip()
        if s:
            return s

    return default


def dedupe_keep_order(items: List[str]) -> List[str]:
    result = []
    seen = set()

    for item in items:
        s = safe_str(item).strip()
        if not s:
            continue

        key = re.sub(r"\s+", " ", s.lower())

        if key not in seen:
            result.append(s)
            seen.add(key)

    return result


def append_jsonl(path: str, item: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


def load_records(input_file: str) -> List[Dict[str, Any]]:
    with open(input_file, "r", encoding="utf-8") as f:
        txt = f.read().strip()

    if not txt:
        raise ValueError("입력 파일이 비어 있습니다.")

    # 일반 JSON
    try:
        raw = json.loads(txt)

        if isinstance(raw, list):
            print(f"[입력 인식] JSON list: {len(raw)}개 record")
            return raw

        if isinstance(raw, dict):
            for key in ["scenarios", "data", "records", "items", "results"]:
                if key in raw and isinstance(raw[key], list):
                    print(f"[입력 인식] wrapper dict '{key}': {len(raw[key])}개 record")
                    return raw[key]

            if "scenario_id" in raw:
                print("[입력 인식] 단일 JSON object: 1개 record")
                return [raw]

            raise ValueError("dict JSON이지만 record list나 scenario_id가 없습니다.")

    except json.JSONDecodeError:
        pass


    records = []

    for line_no, line in enumerate(txt.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
            if isinstance(obj, dict):
                records.append(obj)
            else:
                raise ValueError(f"{line_no}번째 줄이 JSON object가 아닙니다.")
        except Exception as e:
            raise ValueError(f"JSONL 파싱 실패: {line_no}번째 줄 오류: {e}")

    if records:
        print(f"[입력 인식] JSONL: {len(records)}개 record")
        return records

    raise ValueError("입력 파일을 JSON 또는 JSONL로 해석할 수 없습니다.")


def parse_json_loose(text: str) -> Any:
    text = safe_str(text).strip()

    fence = re.search(
        r"```(?:json)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    if fence:
        text = fence.group(1).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    first_obj = text.find("{")
    first_arr = text.find("[")

    starts = []
    if first_obj != -1:
        starts.append((first_obj, "{", "}"))
    if first_arr != -1:
        starts.append((first_arr, "[", "]"))

    if not starts:
        raise ValueError(f"JSON 시작 문자를 찾지 못했습니다. raw={text[:500]}")

    start, open_ch, close_ch = min(starts, key=lambda x: x[0])
    end = text.rfind(close_ch)

    if end <= start:
        raise ValueError(f"JSON 종료 문자를 찾지 못했습니다. raw={text[:800]}")

    sliced = text[start:end + 1]
    return json.loads(sliced)


def parse_chosen_from_response(text: str) -> str:
    obj = parse_json_loose(text)

    if isinstance(obj, dict):
        chosen = safe_str(obj.get("chosen", "")).strip()
        if chosen:
            return normalize_spaces(chosen)

    raise ValueError(f"chosen 필드를 찾지 못했습니다. parsed={obj}")


PLEASANTRY_PATTERNS = [
    r"\bit'?s very smart\b",
    r"\bit'?s good\b",
    r"\byou'?re doing the right thing\b",
    r"\bdon'?t worry\b",
    r"\bi understand\b",
    r"\bi'?m sorry\b",
    r"\bof course\b",
    r"\blet'?s\b",
    r"\bdon'?t hesitate\b",
    r"\bit'?s okay\b",
    r"\bit'?s perfectly fine\b",
    r"\bit'?s always better to be safe than sorry\b",
    r"\btaking these steps will help\b",
    r"\bprotect your money and your personal information\b",
    r"\bgreat job\b",
    r"\bwise of you\b",
    r"\bperfectly understandable\b",
    r"\boh my goodness\b",

    r"걱정하지\s*마세요",
    r"괜찮습니다",
    r"차근차근",
    r"제가\s*도와",
    r"도움이\s*되길",
]

META_PATTERNS = [
    r"\bhere'?s what\b",
    r"\bhere are\b",
    r"\bstep[- ]by[- ]step\b",
    r"\bimportant things to remember\b",
    r"\bremember\b",
    r"\bin summary\b",
    r"\bto summarize\b",
    r"\bthe most important thing\b",
    r"\bfirst\b",
    r"\bnext\b",
    r"\bfinally\b",

    r"아래",
    r"정리",
    r"요약",
    r"기억하세요",
]

SAFETY_KEYWORDS = [
    "bank",
    "card",
    "freeze",
    "block",
    "police",
    "report",
    "password",
    "pin",
    "verification code",
    "id photo",
    "bank details",
    "official number",
    "official channel",
    "do not click",
    "do not install",
    "do not share",
    "transaction",
    "unrecognized",
    "suspicious",
    "link",
    "app",
    "personal information",
    "account",
    "statement",
]

STOPWORDS = {
    "the", "and", "or", "to", "your", "you", "from", "with",
    "for", "if", "of", "in", "on", "a", "an", "is", "are",
    "be", "do", "not", "this", "that", "using", "use", "any",
    "it", "they", "them", "their", "as", "by", "at", "can",
    "could", "should", "would", "will", "when", "what", "how",
    "about", "into", "than", "then", "also", "such",
}


def split_sentences(text: str) -> List[str]:
    text = safe_str(text).strip()

    parts = re.split(r"(?<=[.!?。！？])\s+", text)

    refined = []
    for part in parts:
        refined.extend(re.split(r";\s+", part))

    result = []
    for p in refined:
        p = p.strip()
        if len(p) >= 5:
            result.append(p)

    return result


def tokenize_simple(text: str) -> List[str]:
    text = safe_str(text).lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", text)
    toks = [t for t in text.split() if len(t) >= 2]
    return toks


def jaccard(a: List[str], b: List[str]) -> float:
    sa = set(a)
    sb = set(b)

    if not sa or not sb:
        return 0.0

    return len(sa & sb) / len(sa | sb)


def count_patterns(text: str, patterns: List[str]) -> int:
    low = safe_str(text).lower()
    total = 0

    for pat in patterns:
        total += len(re.findall(pat, low, flags=re.IGNORECASE))

    return total


def redundancy_score_simple(text: str) -> float:
    sents = split_sentences(text)

    if len(sents) <= 1:
        return 0.0

    pairs = 0
    high_sim = 0

    for i in range(len(sents)):
        for j in range(i + 1, len(sents)):
            sim = jaccard(tokenize_simple(sents[i]), tokenize_simple(sents[j]))
            pairs += 1

            if sim >= 0.45:
                high_sim += 1

    if pairs == 0:
        return 0.0

    return clamp((high_sim / pairs) * 2.0)


def length_excess_score(text: str, atomic_actions: List[str]) -> float:
    text = safe_str(text)

    word_count = len(text.split())
    char_count = len(text)

    n_actions = max(1, len(atomic_actions))

    expected_words = max(45, n_actions * 20)
    expected_chars = max(220, n_actions * 85)

    word_excess = max(0.0, (word_count - expected_words) / max(word_count, 1))
    char_excess = max(0.0, (char_count - expected_chars) / max(char_count, 1))

    return clamp((word_excess + char_excess) / 2.0)


def calculate_fluff_index_stable(text: str, atomic_actions: List[str]) -> Dict[str, float]:
    pleasantry_count = count_patterns(text, PLEASANTRY_PATTERNS)
    meta_count = count_patterns(text, META_PATTERNS)

    pleasantry_score = clamp(pleasantry_count / 5.0)
    meta_commentary_score = clamp(meta_count / 6.0)
    redundancy_score = redundancy_score_simple(text)
    nonessential_compression_ratio = length_excess_score(text, atomic_actions)

    fluff_index = (
        0.40 * nonessential_compression_ratio
        + 0.25 * redundancy_score
        + 0.20 * pleasantry_score
        + 0.15 * meta_commentary_score
    )

    return {
        "fluff_index": clamp(fluff_index),
        "nonessential_compression_ratio": clamp(nonessential_compression_ratio),
        "redundancy_score": clamp(redundancy_score),
        "pleasantry_score": clamp(pleasantry_score),
        "meta_commentary_score": clamp(meta_commentary_score),
        "pleasantry_count": pleasantry_count,
        "meta_commentary_count": meta_count,
    }


def extract_atomic_actions_deterministic(source_summary: str) -> List[str]:
    sentences = split_sentences(source_summary)

    actions = []

    for sent in sentences:
        s = sent.strip()
        if not s:
            continue

        s = re.sub(r"^Remember:\s*", "", s, flags=re.IGNORECASE)

        if re.search(r"these steps protect|this caution protects", s, flags=re.IGNORECASE):
            continue

        actions.append(s)

    return dedupe_keep_order(actions)


def extract_safety_critical_actions_deterministic(actions: List[str]) -> List[str]:
    safety = []

    for action in actions:
        low = action.lower()

        if any(k in low for k in SAFETY_KEYWORDS):
            safety.append(action)

    return dedupe_keep_order(safety)


def action_coverage_score(candidate: str, actions: List[str]) -> Dict[str, Any]:
    cand_tokens = set(tokenize_simple(candidate))
    missing = []

    for action in actions:
        action_tokens = [
            t for t in tokenize_simple(action)
            if t not in STOPWORDS
        ]

        if not action_tokens:
            continue

        overlap = len(set(action_tokens) & cand_tokens) / max(1, len(set(action_tokens)))
        if overlap < 0.30:
            missing.append(action)

    total = max(1, len(actions))
    coverage = 1.0 - (len(missing) / total)

    return {
        "coverage_score": clamp(coverage),
        "missing_actions": missing,
    }


def build_eval_object(text: str, atomic_actions: List[str], safety_actions: List[str]) -> Dict[str, Any]:
    coverage_obj = action_coverage_score(text, atomic_actions)

    if safety_actions:
        safety_obj = action_coverage_score(text, safety_actions)
    else:
        safety_obj = {
            "coverage_score": 1.0,
            "missing_actions": []
        }

    fluff_obj = calculate_fluff_index_stable(text, atomic_actions)

    safety_critical_missing = len(safety_obj["missing_actions"]) > 0

    faithful = (
        coverage_obj["coverage_score"] >= MIN_CHOSEN_COVERAGE
        and not safety_critical_missing
    )

    return {
        "coverage_score": coverage_obj["coverage_score"],
        "missing_actions": coverage_obj["missing_actions"],
        "added_external_facts": [],
        "safety_critical_missing": safety_critical_missing,
        "safety_missing_actions": safety_obj["missing_actions"],
        "faithful": faithful,

        "nonessential_compression_ratio": fluff_obj["nonessential_compression_ratio"],
        "redundancy_score": fluff_obj["redundancy_score"],
        "pleasantry_score": fluff_obj["pleasantry_score"],
        "meta_commentary_score": fluff_obj["meta_commentary_score"],
        "pleasantry_count": fluff_obj["pleasantry_count"],
        "meta_commentary_count": fluff_obj["meta_commentary_count"],
        "fluff_index": fluff_obj["fluff_index"],
    }

def model_supports_thinking_config(model_name: str) -> bool:
    return "2.5" in model_name or "3" in model_name


def make_generation_config(model_name: str) -> types.GenerateContentConfig:
    kwargs = {
        "temperature": 0.25,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "response_mime_type": "application/json",
    }


    if model_supports_thinking_config(model_name):
        kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=0)

    return types.GenerateContentConfig(**kwargs)


def build_generation_prompt(
    user_prompt: str,
    persona_context: str,
    answer_system_instruction: str,
    source_summary: str,
    atomic_actions: List[str],
) -> str:
    actions_json = json.dumps(atomic_actions, ensure_ascii=False, indent=2)

    prompt = f"""
Return only valid JSON with one field: "chosen".

Create a concise alternative answer for a DPO preference dataset.

Task:
Write one answer to the user. Preserve every required action. Remove unnecessary fluff.

Rules:
- Preserve every item in REQUIRED_ACTIONS.
- Do not copy SOURCE_SUMMARY verbatim.
- Do not add new facts, URLs, phone numbers, agencies, or unsupported steps.
- Do not omit bank contact, card freeze/block, police report, password change, account checking, suspicious-link warning, unknown-app warning, or sensitive-information warning when present.
- No greeting.
- No praise, reassurance, apology, emotional validation, or closing pleasantry.
- Avoid "Here is", "Let's", "In summary", "Remember", "The most important thing".
- Use simple English.
- Keep offline help when present: call bank, visit bank branch, visit police station, ask trusted person, ask community center.
- 80 to 160 words is preferred.

USER_PROMPT:
{user_prompt}

PERSONA:
{persona_context}

SYSTEM_INSTRUCTION:
{answer_system_instruction}

SOURCE_SUMMARY:
{source_summary}

REQUIRED_ACTIONS:
{actions_json}

JSON format:
{{"chosen": "your concise alternative answer"}}
""".strip()

    return prompt


def generate_chosen_with_fallback(
    user_prompt: str,
    persona_context: str,
    answer_system_instruction: str,
    source_summary: str,
    atomic_actions: List[str],
) -> Tuple[str, str, List[Dict[str, Any]], str]:
    """
    모델 fallback + retry + JSON parse까지 한 함수 안에서 처리.
    API 응답은 받았지만 JSON parse가 실패한 경우도 실패로 보고 다음 attempt/model로 넘어감.
    """
    prompt = build_generation_prompt(
        user_prompt=user_prompt,
        persona_context=persona_context,
        answer_system_instruction=answer_system_instruction,
        source_summary=source_summary,
        atomic_actions=atomic_actions,
    )

    attempt_logs = []
    last_error = None

    for model_name in MODEL_CANDIDATES:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            raw_text = ""

            try:
                config = make_generation_config(model_name)

                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=config,
                )

                raw_text = safe_str(getattr(response, "text", "")).strip()

                if not raw_text:
                    raise ValueError("빈 응답이 반환되었습니다.")

                chosen = parse_chosen_from_response(raw_text)

                attempt_logs.append({
                    "model": model_name,
                    "attempt": attempt,
                    "status": "success",
                    "error": "",
                    "raw_length": len(raw_text),
                    "raw_preview": raw_text[:500],
                })

                return chosen, model_name, attempt_logs, raw_text[:1000]

            except Exception as e:
                last_error = e
                err = str(e)

                attempt_logs.append({
                    "model": model_name,
                    "attempt": attempt,
                    "status": "failed",
                    "error": err[:1000],
                    "raw_length": len(raw_text),
                    "raw_preview": raw_text[:500],
                })

                print(
                    f"   [API/Parse 실패] "
                    f"model={model_name}, attempt={attempt}/{MAX_RETRIES_PER_MODEL}: "
                    f"{err[:300]}"
                )

                if attempt < MAX_RETRIES_PER_MODEL:
                    sleep_s = min(
                        MAX_RETRY_SLEEP,
                        BASE_RETRY_SLEEP * (2 ** (attempt - 1)) + random.uniform(0, 1.0)
                    )
                    print(f"   -> {sleep_s:.1f}초 대기 후 재시도")
                    time.sleep(sleep_s)

        print(f"   -> model={model_name} 실패. 다음 모델 시도.")

    raise RuntimeError(f"모든 모델 호출/파싱 실패: {last_error}")


def clean_p1_fallback(source_summary: str) -> str:
    text = safe_str(source_summary).strip()

    text = re.sub(r"\bRemember:\s*", "", text, flags=re.IGNORECASE)

    text = re.sub(
        r"\s*These steps protect your money(?: and information| and personal information)?\.?$",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*This caution protects your money(?: and information| and personal information)?\.?$",
        "",
        text,
        flags=re.IGNORECASE
    )

    return normalize_spaces(text)


def get_scenario_id(row: pd.Series, index: int) -> str:
    sid = first_available(row, ["scenario_id", "id", "scenarioId"], default="")
    return sid if sid else f"row_{index}"


def get_prompt_number(row: pd.Series, index: int) -> Any:
    val = get_nested(row, "prompt_number", default=None)

    if val is None or safe_str(val).strip() == "":
        return index + 1

    return val


def get_record_id(row: pd.Series, index: int) -> str:
    scenario_id = get_scenario_id(row, index)
    prompt_number = get_prompt_number(row, index)
    return f"{scenario_id}_prompt_{prompt_number}"


def get_user_prompt(row: pd.Series) -> str:
    return first_available(
        row,
        ["user_prompt", "prompt", "input", "question", "universal_question"],
        default=""
    )


def get_persona(row: pd.Series) -> str:
    return first_available(
        row,
        ["persona_context", "persona", "persona_type"],
        default="General User"
    )


def get_answer_system_instruction(row: pd.Series) -> str:
    return first_available(
        row,
        ["answer_system_instruction", "system_instruction", "instruction"],
        default=""
    )


def get_source_summary(row: pd.Series) -> str:
    return first_available(
        row,
        [
            "compressed_responses.p1_abstractive_text",
            "p1_abstractive_text",
            "source_summary",
            "summary",
        ],
        default=""
    )


def get_original_response(row: pd.Series) -> str:
    return first_available(
        row,
        ["original_response", "response", "answer", "model_response"],
        default=""
    )


def get_scenario_type(row: pd.Series, scenario_id: str) -> str:
    explicit = first_available(
        row,
        ["fraud_type", "scenario_type", "type", "category"],
        default=""
    )

    if explicit:
        return explicit

    return re.sub(r"_\d+$", "", scenario_id)


def get_original_fluff_index(row: pd.Series) -> Optional[float]:
    val = get_nested(row, "metrics.fluff_index", default=None)

    if val is None or safe_str(val).strip() == "":
        return None

    try:
        return float(val)
    except Exception:
        return None


def get_token_count(row: pd.Series, key: str) -> Optional[int]:
    val = get_nested(row, f"metrics.token_counts.{key}", default=None)

    if val is None or safe_str(val).strip() == "":
        return None

    try:
        return int(val)
    except Exception:
        return None



def process_row(row: pd.Series, index: int) -> Tuple[Optional[Dict[str, Any]], Optional[Dict[str, Any]]]:
    scenario_id = get_scenario_id(row, index)
    prompt_number = get_prompt_number(row, index)
    record_id = get_record_id(row, index)

    scenario_type = get_scenario_type(row, scenario_id)

    user_prompt = get_user_prompt(row)
    persona_context = get_persona(row)
    answer_system_instruction = get_answer_system_instruction(row)

    source_summary = get_source_summary(row)
    original_response = get_original_response(row)
    input_original_fluff = get_original_fluff_index(row)

    print("\n" + "=" * 80)
    print(f"[작업 index={index}, record_id={record_id}]")
    print(f"scenario_id={scenario_id} | prompt_number={prompt_number} | type={scenario_type}")

    if not source_summary:
        error = {
            "index": index,
            "record_id": record_id,
            "scenario_id": scenario_id,
            "prompt_number": prompt_number,
            "stage": "data_validation",
            "error_type": "missing_source_summary",
            "error_message": "compressed_responses.p1_abstractive_text가 없음",
            "created_at": datetime.now().isoformat(timespec="seconds"),
        }
        print("  -> source_summary 없음. skip")
        return None, error

    if not original_response:
        error = {
            "index": index,
            "record_id": record_id,
            "scenario_id": scenario_id,
            "prompt_number": prompt_number,
            "stage": "data_validation",
            "error_type": "missing_original_response",
            "error_message": "original_response가 없음",
            "created_at": datetime.now().isoformat(timespec="seconds"),
        }
        print("  -> original_response 없음. skip")
        return None, error

    atomic_actions = extract_atomic_actions_deterministic(source_summary)
    safety_actions = extract_safety_critical_actions_deterministic(atomic_actions)

    atomic_actions = dedupe_keep_order(atomic_actions + safety_actions)

    print(f"  atomic_actions={len(atomic_actions)}, safety_actions={len(safety_actions)}")

    chosen_source = "api_generated"
    used_model = None
    api_error = None
    attempt_logs = []
    raw_api_response_preview = ""

    try:
        chosen_text, used_model, attempt_logs, raw_api_response_preview = generate_chosen_with_fallback(
            user_prompt=user_prompt,
            persona_context=persona_context,
            answer_system_instruction=answer_system_instruction,
            source_summary=source_summary,
            atomic_actions=atomic_actions,
        )

    except Exception as e:
        api_error = str(e)
        print(f"  -> API chosen 생성 실패: {api_error[:500]}")

        if FALLBACK_TO_P1_ON_API_ERROR:
            chosen_text = clean_p1_fallback(source_summary)
            chosen_source = "fallback_p1_after_api_failure"
            used_model = None
            print("  -> fallback p1 chosen 사용")
        else:
            error = {
                "index": index,
                "record_id": record_id,
                "scenario_id": scenario_id,
                "prompt_number": prompt_number,
                "stage": "api_generation",
                "error_type": "api_or_parse_failure",
                "error_message": api_error[:2000],
                "attempt_logs": json.dumps(attempt_logs, ensure_ascii=False),
                "created_at": datetime.now().isoformat(timespec="seconds"),
            }
            print("  -> skip")
            return None, error

    rejected_text = original_response

    chosen_eval = build_eval_object(chosen_text, atomic_actions, safety_actions)
    rejected_eval = build_eval_object(rejected_text, atomic_actions, safety_actions)

    fluff_delta = rejected_eval["fluff_index"] - chosen_eval["fluff_index"]

    print(f"  used_model={used_model}")
    print(f"  chosen_fluff={chosen_eval['fluff_index']:.3f}")
    print(f"  rejected_fluff={rejected_eval['fluff_index']:.3f}")
    print(f"  delta={fluff_delta:.3f}")
    print(f"  chosen_coverage={chosen_eval['coverage_score']:.3f}")
    print(f"  rejected_coverage={rejected_eval['coverage_score']:.3f}")
    print(f"  chosen_words={len(chosen_text.split())}, rejected_words={len(rejected_text.split())}")

    if chosen_eval["coverage_score"] < MIN_CHOSEN_COVERAGE:
        print("  -> 경고: chosen coverage가 낮습니다. CSV에서 검토 필요.")

    item = {
        "record_id": record_id,
        "source_index": index,
        "scenario_id": scenario_id,
        "prompt_number": prompt_number,
        "scenario_type": scenario_type,
        "persona_type": first_available(row, ["persona_type"], default=""),
        "persona": persona_context,

        "prompt": user_prompt,
        "chosen": chosen_text,
        "rejected": rejected_text,

        "source_summary": source_summary,
        "atomic_actions": atomic_actions,
        "safety_critical_actions": safety_actions,

        "chosen_source": chosen_source,
        "rejected_source": "original_response",
        "generation_model": used_model,
        "api_error": api_error,
        "api_attempt_logs": attempt_logs,
        "raw_api_response_preview": raw_api_response_preview,

        "metrics": {
            "input_original_fluff_index": input_original_fluff,
            "input_token_count_original": get_token_count(row, "original"),
            "input_token_count_p1_abstractive": get_token_count(row, "p1_abstractive"),
            "input_token_count_p2_extractive": get_token_count(row, "p2_extractive"),
            "input_token_count_p3_word_removed": get_token_count(row, "p3_word_removed"),

            "chosen_fluff_index": chosen_eval["fluff_index"],
            "rejected_fluff_index": rejected_eval["fluff_index"],
            "fluff_delta_rejected_minus_chosen": fluff_delta,

            "chosen_coverage_score": chosen_eval["coverage_score"],
            "rejected_coverage_score": rejected_eval["coverage_score"],

            "chosen_missing_actions": chosen_eval["missing_actions"],
            "rejected_missing_actions": rejected_eval["missing_actions"],

            "chosen_safety_critical_missing": chosen_eval["safety_critical_missing"],
            "rejected_safety_critical_missing": rejected_eval["safety_critical_missing"],

            "chosen_eval": chosen_eval,
            "rejected_eval": rejected_eval,
        },

        "created_at": datetime.now().isoformat(timespec="seconds"),
    }

    return item, None


def run_pipeline():
    if CLEAR_OLD_OUTPUT_AT_START:
        for path in [OUTPUT_FILE, REPORT_FILE, ERROR_LOG_FILE]:
            if os.path.exists(path):
                os.remove(path)
                print(f"[삭제] 기존 파일 삭제: {path}")

    print(f"데이터 파일 로딩: {INPUT_FILE}")

    records = load_records(INPUT_FILE)
    df = pd.DataFrame(records)

    print(f"DataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    if "prompt_number" not in df.columns:
        raise ValueError("입력 파일에 prompt_number 컬럼이 없습니다.")

    df["_prompt_number_numeric"] = pd.to_numeric(df["prompt_number"], errors="coerce")

    df_run = df[
        (df["_prompt_number_numeric"] >= PROMPT_NUMBER_START)
        & (df["_prompt_number_numeric"] <= PROMPT_NUMBER_END)
    ].copy()

    df_run = df_run.sort_values("_prompt_number_numeric").copy()

    print("\n" + "=" * 80)
    print("One-call API Fluff-reduced DPO 파이프라인 시작")
    print(f"전체 record 수: {len(df)}")
    print(f"처리 기준: prompt_number {PROMPT_NUMBER_START}~{PROMPT_NUMBER_END}")
    print(f"대상 row 수: {len(df_run)}")
    print(f"대상 prompt_number 목록:")
    print(df_run['_prompt_number_numeric'].tolist())
    print(f"모델 fallback 순서: {MODEL_CANDIDATES}")
    print("2.5 계열 thinking_budget=0 적용")
    print("=" * 80)

    if len(df_run) == 0:
        raise ValueError(
            f"prompt_number {PROMPT_NUMBER_START}~{PROMPT_NUMBER_END}에 해당하는 row가 없습니다."
        )

    results = []
    report_rows = []
    error_rows = []

    for index, row in df_run.iterrows():
        try:
            item, error = process_row(row, index)

            if error is not None:
                error_rows.append(error)
                pd.DataFrame(error_rows).to_csv(
                    ERROR_LOG_FILE,
                    index=False,
                    encoding="utf-8-sig"
                )
                time.sleep(SLEEP_BETWEEN_ROWS)
                continue

            if item is None:
                time.sleep(SLEEP_BETWEEN_ROWS)
                continue

            append_jsonl(OUTPUT_FILE, item)
            results.append(item)

            report_rows.append({
                "source_index": item["source_index"],
                "record_id": item["record_id"],
                "scenario_id": item["scenario_id"],
                "prompt_number": item["prompt_number"],
                "scenario_type": item["scenario_type"],
                "persona_type": item["persona_type"],

                "chosen_source": item["chosen_source"],
                "generation_model": item["generation_model"],

                "input_original_fluff_index": item["metrics"]["input_original_fluff_index"],
                "chosen_fluff_index": item["metrics"]["chosen_fluff_index"],
                "rejected_fluff_index": item["metrics"]["rejected_fluff_index"],
                "fluff_delta": item["metrics"]["fluff_delta_rejected_minus_chosen"],

                "chosen_coverage_score": item["metrics"]["chosen_coverage_score"],
                "rejected_coverage_score": item["metrics"]["rejected_coverage_score"],

                "chosen_safety_critical_missing": item["metrics"]["chosen_safety_critical_missing"],
                "rejected_safety_critical_missing": item["metrics"]["rejected_safety_critical_missing"],

                "num_atomic_actions": len(item["atomic_actions"]),
                "num_safety_actions": len(item["safety_critical_actions"]),

                "chosen_word_count": len(item["chosen"].split()),
                "rejected_word_count": len(item["rejected"].split()),
            })

            pd.DataFrame(report_rows).to_csv(
                REPORT_FILE,
                index=False,
                encoding="utf-8-sig"
            )

        except Exception as e:
            err = {
                "index": index,
                "record_id": get_record_id(row, index),
                "scenario_id": get_scenario_id(row, index),
                "prompt_number": get_prompt_number(row, index),
                "stage": "unexpected_exception",
                "error_type": type(e).__name__,
                "error_message": str(e)[:2000],
                "traceback": traceback.format_exc()[:4000],
                "created_at": datetime.now().isoformat(timespec="seconds"),
            }

            error_rows.append(err)

            pd.DataFrame(error_rows).to_csv(
                ERROR_LOG_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print(f"\n[예상 밖 오류] index={index}: {e}")

        time.sleep(SLEEP_BETWEEN_ROWS)

    report_df = pd.DataFrame(report_rows)
    error_df = pd.DataFrame(error_rows)

    print("\n" + "=" * 80)
    print("파이프라인 완료")
    print(f"시도 대상 row 수: {len(df_run)}")
    print(f"성공 DPO pair 수: {len(results)}")
    print(f"실패/skip 수: {len(error_rows)}")
    print(f"성공률: {len(results) / max(1, len(df_run)) * 100:.1f}%")
    print(f"JSONL 저장 위치: {OUTPUT_FILE}")
    print(f"CSV 리포트 저장 위치: {REPORT_FILE}")
    print(f"Error 로그 저장 위치: {ERROR_LOG_FILE}")

    if len(report_df) > 0:
        print("\n[요약 통계]")
        print(f"평균 chosen fluff:       {report_df['chosen_fluff_index'].mean():.3f}")
        print(f"평균 rejected fluff:     {report_df['rejected_fluff_index'].mean():.3f}")
        print(f"평균 fluff delta:        {report_df['fluff_delta'].mean():.3f}")
        print(f"평균 chosen coverage:    {report_df['chosen_coverage_score'].mean():.3f}")
        print(f"평균 rejected coverage:  {report_df['rejected_coverage_score'].mean():.3f}")

        print("\n[리포트 미리보기]")
        display(report_df)

    if len(error_df) > 0:
        print("\n[실패 로그 미리보기]")
        display(error_df)

    print("=" * 80)

    return results, report_df, error_df


results, report_df, error_df = run_pipeline()